# EDA conjunto — Data4Cyber (S0–S6 + S1-alt)

Análisis exploratorio sobre las **8 tablas de proceso** del release, unidas en un solo
`DataFrame` con la columna `escenario`. Continúa la línea de `EDA_S0`, `EDA_S1` y `EDA_S6`.

**Cómo se ejecuta.** La celda 0 resuelve sola la ruta de datos: monta Drive si corre en
Colab y cae a `datasets/` si corre local. No hay rutas absolutas en ninguna otra celda.

**Convención del archivo.** Una celda = un paso verificable. Cada bloque de código lleva
arriba el comentario de para qué está, y toda salida impresa va etiquetada.


In [7]:
# === 0. Configuración y carga de los escenarios ===

%matplotlib inline
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)

# --- único punto con la ruta de datos: Drive en Colab, carpeta local en VS Code ---
# etiqueta legible -> archivo; S1-alt no sigue la regla dataset<etiqueta>.csv, por eso es explícito
ARCHIVOS = {
    "S0":     "datasetS0.csv",
    "S1":     "datasetS1.csv",
    "S1-alt": "datasetS1alt.csv",
    "S2":     "datasetS2.csv",
    "S3":     "datasetS3.csv",
    "S4":     "datasetS4.csv",
    "S5":     "datasetS5.csv",
}

try:
    from google.colab import drive
    drive.mount("/content/gdrive/")
    BASE = "/content/gdrive/MyDrive/TrabajoFinal/datasets/"
except ImportError:
    BASE = "datasets/"

# --- unión: cada fila conserva de qué escenario viene antes de concatenar ---
partes = []
for esc, archivo in ARCHIVOS.items():
    ruta = os.path.join(BASE, archivo)
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"falta {ruta} | BASE={BASE!r}")
    parte = pd.read_csv(ruta)
    # la columna va primera para que se lea el origen sin scrollear a la derecha
    parte.insert(0, "escenario", esc)
    partes.append(parte)

df = pd.concat(partes, ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# --- control de la unión: filas por escenario y columnas que no existen en todos ---
print("filas por escenario:")
print(df["escenario"].value_counts().reindex(ARCHIVOS).to_string())
print(f"\nfilas totales: {len(df):,} | columnas: {df.shape[1]}")

# concat alinea por nombre: si un escenario no trae una columna, queda en NaN en ese bloque
ausentes = [c for c in df.columns
            if df.groupby("escenario")[c].apply(lambda s: s.isna().all()).any()]
print("columnas ausentes en algún escenario:", ausentes)

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
filas por escenario:
escenario
S0        1798
S1        1798
S1-alt    1792
S2        1768
S3        1800
S4        1798
S5        1800

filas totales: 12,554 | columnas: 156
columnas ausentes en algún escenario: []


In [2]:
# --- agrupación de columnas: dispositivo, tipo, fase ---
SIG_TIPO = {
    "uln1": "Tensión fase-neutro", "uln2": "Tensión fase-neutro", "uln3": "Tensión fase-neutro",
    "ull12": "Tensión fase-fase", "ull23": "Tensión fase-fase", "ull31": "Tensión fase-fase",
    "iln1": "Corriente de fase", "iln2": "Corriente de fase", "iln3": "Corriente de fase",
    "i_sum3": "Corriente de neutro",
    "pln1": "Potencia activa", "pln2": "Potencia activa", "pln3": "Potencia activa", "p_sum3": "Potencia activa",
    "sln1": "Potencia aparente", "sln2": "Potencia aparente", "sln3": "Potencia aparente", "s_sum3": "Potencia aparente",
    "qln1": "Potencia reactiva", "qln2": "Potencia reactiva", "qln3": "Potencia reactiva", "q_sum3": "Potencia reactiva",
    "cos_phi1": "Factor de potencia", "cos_phi2": "Factor de potencia", "cos_phi3": "Factor de potencia",
    "freq": "Frecuencia",
    "phase_seq": "Secuencia de fases",
    "realtime": "Marca de tiempo del equipo",
}
UNIDAD = {
    "Tensión fase-neutro": "V", "Tensión fase-fase": "V", "Corriente de fase": "A",
    "Corriente de neutro": "A", "Potencia activa": "W", "Potencia aparente": "VA",
    "Potencia reactiva": "var", "Factor de potencia": "—", "Frecuencia": "Hz",
    "Secuencia de fases": "—", "Marca de tiempo del equipo": "s",
    "SOC": "fracción 0-1", "Señal de precio": "EUR/kWh", "Contexto de escenario": "—",
    "Control / etiquetas": "—", "Desconocido": "?",
}
FASE = {"ull12": "L1-L2", "ull23": "L2-L3", "ull31": "L3-L1"}

def partir(c):
    if "." in c:
        dev, sig = c.split(".", 1)
    else:
        dev, sig = "(sin prefijo)", c
    return dev, sig

def tipo_de(dev, sig):
    if dev == "(sin prefijo)":
        return "Control / etiquetas"
    if dev.startswith("Battery-SOC"):
        return "SOC"
    if dev == "MQTT-Price-Signal":
        return "Señal de precio"
    if dev == "Profile":
        return "Contexto de escenario"
    return SIG_TIPO.get(sig, "Desconocido")

def fase_de(sig):
    if sig in FASE:
        return FASE[sig]
    if sig.endswith("sum3"):
        return "trifásica"
    if sig.endswith("1"):
        return "L1"
    if sig.endswith("2"):
        return "L2"
    if sig.endswith("3"):
        return "L3"
    return "—"

reg = []
for c in df.columns:
    dev, sig = partir(c)
    t = tipo_de(dev, sig)
    reg.append({"columna": c, "dispositivo": dev, "señal": sig,
                "tipo": t, "unidad": UNIDAD.get(t, "?"), "fase": fase_de(sig)})
cols = pd.DataFrame(reg)

print("columnas:", len(cols), "| dispositivos:", cols.dispositivo.nunique(), "| tipos:", cols.tipo.nunique())
print("\n--- por dispositivo ---")
por_dev = (cols.groupby("dispositivo")
           .agg(columnas=("columna", "size"), tipos=("tipo", lambda s: ", ".join(sorted(set(s)))))
           .sort_values("columnas", ascending=False))
display(por_dev)

columnas: 157 | dispositivos: 10 | tipos: 15

--- por dispositivo ---


,columnas,tipos
dispositivo,,
BSS-Inverter-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Load-B-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Substation-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Load-A-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
PV-Inverter-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Profile,8,Contexto de escenario
(sin prefijo),5,Control / etiquetas
MQTT-Price-Signal,2,Señal de precio
Battery-SOC-Reference,1,SOC


## 2. Datos Nulos

En este release **una celda vacía no es un cero ni un `NaN` imputado: es ausencia de valor de origen**.

In [3]:
# --- vacíos por columna y por fila ---

num = df.select_dtypes(include=[np.number])
no_num = [c for c in df.columns if c not in num.columns]
miss = df.isna()

print(f"columnas numéricas: {len(num.columns)} | no numéricas: {len(no_num)}")
if no_num:
    print("no numéricas:", no_num)

print(f"celdas vacías: {int(miss.values.sum()):,} de {miss.size:,} ({miss.values.mean()*100:.2f} %)")
print(f"filas con algún vacío: {int((miss.sum(axis=1) > 0).sum()):,} de {len(miss):,}")

por_col = miss.mean().mul(100).sort_values(ascending=False)
con_vacios = por_col[por_col > 0]
print(f"ncolumnas con al menos un vacío: {len(con_vacios)} de {len(por_col)}")
print("\ntop 15 columnas más incompletas (%):")
print(con_vacios.head(15).round(2).to_string())


columnas numéricas: 150 | no numéricas: 7
no numéricas: ['escenario', 'timestamp', 'attack_active', 'attack_phase', 'attack_phase_all', 'Profile.timestamp', 'MQTT-Price-Signal.event']
celdas vacías: 20,123 de 2,253,578 (0.89 %)
filas con algún vacío: 12,992 de 14,354
ncolumnas con al menos un vacío: 152 de 157

top 15 columnas más incompletas (%):
MQTT-Price-Signal.event                90.51
MQTT-Price-Signal.price_eur_per_kwh     6.77
Battery-SOC-Reported.soc_value          1.40
Profile.total_irradiance                1.37
Profile.pv_active_kw                    1.37
Profile.load1_kw                        1.37
Profile.pv_input_kw                     1.37
Profile.load2_kw                        1.37
Profile.ambient_temperature             1.37
Profile.pv_reactive_kvar                1.37
Profile.timestamp                       1.37
Battery-SOC-Reference.soc_value         0.35
BSS-Inverter-AC-Meter.i_sum3            0.22
BSS-Inverter-AC-Meter.qln3              0.22
BSS-Inverter-AC-Mete

 # Eliminar columnas sin aporte
 por ahora vamos a descartar las columas que involucren marcas temporales y etapas del ataque ya que no estamos interezados en analizar estas caracteristicas.
"

In [4]:
print(df.shape)
df = df.drop(columns=["attack_phase", "attack_phase_all","Profile.timestamp","timestamp"])
df = df.drop(columns=[c for c in df.columns if "realtime" in c ])
df = df.drop(columns=[c for c in df.columns if "phase_seq" in c ])
print(df.shape)


(14354, 157)
(14354, 143)


In [5]:
# Identificar columnas con al menos un valor nulo
columnas_con_nulos = df.columns[df.isna().any()]


# Filtrar filas que contienen nulos y mostrar solo esas columnas
filas_nulas_reducido = df.loc[df.isna().any(axis=1), columnas_con_nulos]
filas_nulas_reducido.head()

,BSS-Inverter-AC-Meter.cos_phi1,BSS-Inverter-AC-Meter.cos_phi2,BSS-Inverter-AC-Meter.cos_phi3,BSS-Inverter-AC-Meter.freq,BSS-Inverter-AC-Meter.i_sum3,BSS-Inverter-AC-Meter.iln1,BSS-Inverter-AC-Meter.iln2,BSS-Inverter-AC-Meter.iln3,BSS-Inverter-AC-Meter.p_sum3,BSS-Inverter-AC-Meter.pln1,BSS-Inverter-AC-Meter.pln2,BSS-Inverter-AC-Meter.pln3,BSS-Inverter-AC-Meter.q_sum3,BSS-Inverter-AC-Meter.qln1,BSS-Inverter-AC-Meter.qln2,BSS-Inverter-AC-Meter.qln3,BSS-Inverter-AC-Meter.s_sum3,BSS-Inverter-AC-Meter.sln1,BSS-Inverter-AC-Meter.sln2,BSS-Inverter-AC-Meter.sln3,BSS-Inverter-AC-Meter.ull12,BSS-Inverter-AC-Meter.ull23,BSS-Inverter-AC-Meter.ull31,BSS-Inverter-AC-Meter.uln1,BSS-Inverter-AC-Meter.uln2,BSS-Inverter-AC-Meter.uln3,Battery-SOC-Reference.soc_value,Battery-SOC-Reported.soc_value,Load-A-AC-Meter.cos_phi1,Load-A-AC-Meter.cos_phi2,...,Profile.pv_input_kw,Profile.pv_reactive_kvar,Profile.total_irradiance,Substation-AC-Meter.cos_phi1,Substation-AC-Meter.cos_phi2,Substation-AC-Meter.cos_phi3,Substation-AC-Meter.freq,Substation-AC-Meter.i_sum3,Substation-AC-Meter.iln1,Substation-AC-Meter.iln2,Substation-AC-Meter.iln3,Substation-AC-Meter.p_sum3,Substation-AC-Meter.pln1,Substation-AC-Meter.pln2,Substation-AC-Meter.pln3,Substation-AC-Meter.q_sum3,Substation-AC-Meter.qln1,Substation-AC-Meter.qln2,Substation-AC-Meter.qln3,Substation-AC-Meter.s_sum3,Substation-AC-Meter.sln1,Substation-AC-Meter.sln2,Substation-AC-Meter.sln3,Substation-AC-Meter.ull12,Substation-AC-Meter.ull23,Substation-AC-Meter.ull31,Substation-AC-Meter.uln1,Substation-AC-Meter.uln2,Substation-AC-Meter.uln3,MQTT-Price-Signal.event
0,-0.310502,-0.703539,-0.989501,50.008049,0.101279,0.39492,0.377298,0.421309,-53.832207,-18.320284,-16.559868,-18.952057,28.835691,-2.792666,-9.061968,3.146089,281.451599,93.153397,88.853355,99.444839,407.938171,408.500793,408.841309,235.879349,235.498993,236.037918,NaN,NaN,-0.778859,0.997305,...,NaN,NaN,NaN,-0.965705,-0.966816,-0.962146,50.005825,1.437780,9.847172,9.582252,9.602752,-5902.140137,-1999.145142,-1944.466309,-1958.528809,1616.824585,541.077271,517.283081,558.464233,6802.083984,2306.897217,2242.235840,2252.951416,405.437134,405.948944,406.069855,234.270035,233.998840,234.615189,NaN
1,-0.310502,-0.703539,-0.989501,50.008049,0.101279,0.39492,0.377298,0.421309,-53.832207,-18.320284,-16.559868,-18.952057,28.835691,-2.792666,-9.061968,3.146089,281.451599,93.153397,88.853355,99.444839,407.938171,408.500793,408.841309,235.879349,235.498993,236.037918,0.199597,NaN,-0.778859,0.997305,...,NaN,NaN,NaN,-0.965705,-0.966816,-0.962146,50.005825,1.437780,9.847172,9.582252,9.602752,-5902.140137,-1999.145142,-1944.466309,-1958.528809,1616.824585,541.077271,517.283081,558.464233,6802.083984,2306.897217,2242.235840,2252.951416,405.437134,405.948944,406.069855,234.270035,233.998840,234.615189,NaN
2,-0.310502,-0.703539,-0.989501,50.008049,0.101279,0.39492,0.377298,0.421309,-53.832207,-18.320284,-16.559868,-18.952057,28.835691,-2.792666,-9.061968,3.146089,281.451599,93.153397,88.853355,99.444839,407.938171,408.500793,408.841309,235.879349,235.498993,236.037918,0.199597,NaN,-0.778859,0.997305,...,NaN,NaN,NaN,-0.965705,-0.966816,-0.962146,50.005825,1.437780,9.847172,9.582252,9.602752,-5902.140137,-1999.145142,-1944.466309,-1958.528809,1616.824585,541.077271,517.283081,558.464233,6802.083984,2306.897217,2242.235840,2252.951416,405.437134,405.948944,406.069855,234.270035,233.998840,234.615189,NaN
3,-0.310502,-0.703539,-0.989501,50.008049,0.101279,0.39492,0.377298,0.421309,-53.832207,-18.320284,-16.559868,-18.952057,28.835691,-2.792666,-9.061968,3.146089,281.451599,93.153397,88.853355,99.444839,407.938171,408.500793,408.841309,235.879349,235.498993,236.037918,0.199597,NaN,-0.778859,0.997305,...,NaN,NaN,NaN,-0.965705,-0.966816,-0.962146,50.005825,1.437780,9.847172,9.582252,9.602752,-5902.140137,-1999.145142,-1944.466309,-1958.528809,1616.824585,541.077271,517.283081,558.464233,6802.083984,2306.897217,2242.235840,2252.951416,405.437134,405.948944,406.069855,234.2700

Imputacion de datos nulos por la media

In [6]:
# 1. Irradiancia total
media_irradiance = df['Profile.total_irradiance'].mean()
df['Profile.total_irradiance'] = df['Profile.total_irradiance'].fillna(media_irradiance)

# 2. Temperatura ambiente
media_temp = df['Profile.ambient_temperature'].mean()
df['Profile.ambient_temperature'] = df['Profile.ambient_temperature'].fillna(media_temp)

# 3. Señal de precio MQTT
media_price = df['MQTT-Price-Signal.price_eur_per_kwh'].mean()
df['MQTT-Price-Signal.price_eur_per_kwh'] = df['MQTT-Price-Signal.price_eur_per_kwh'].fillna(media_price)

# 4. SOC de referencia de batería
media_soc_ref = df['Battery-SOC-Reference.soc_value'].mean()
df['Battery-SOC-Reference.soc_value'] = df['Battery-SOC-Reference.soc_value'].fillna(media_soc_ref)

# 5. SOC reportado de batería
media_soc_rep = df['Battery-SOC-Reported.soc_value'].mean()
df['Battery-SOC-Reported.soc_value'] = df['Battery-SOC-Reported.soc_value'].fillna(media_soc_rep)

# 6. Carga de perfil 1
media_load1 = df['Profile.load1_kw'].mean()
df['Profile.load1_kw'] = df['Profile.load1_kw'].fillna(media_load1)

# 7. Carga de perfil 2
media_load2 = df['Profile.load2_kw'].mean()
df['Profile.load2_kw'] = df['Profile.load2_kw'].fillna(media_load2)

media_active_kw = df['Profile.pv_active_kw'].mean()
df['Profile.pv_active_kw'] = df['Profile.pv_active_kw'].fillna(media_active_kw)

media_input_kw = df['Profile.pv_input_kw'].mean()
df['Profile.pv_input_kw'] = df['Profile.pv_input_kw'].fillna(media_input_kw)

media_reactive_kvar = df['Profile.pv_reactive_kvar'].mean()
df['Profile.pv_reactive_kvar'] = df['Profile.pv_reactive_kvar'].fillna(media_reactive_kvar)
